## Mart_Table (mart_seller_scorecard) GOLD LAYER INSERTION

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

### Importing Libraries

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, avg, count, countDistinct, max as spark_max,
    round as spark_round, col, when, rank, percent_rank, try_divide, row_number
)
from pyspark.sql import Window


What Here We DO:

- Groups by seller_id
- Calculates revenue, on-time rate, avg rating
- Normalizes each metric to 0-100 scale
- Creates composite seller_health_score = (on_time × 0.40) + (rating × 0.30) + (revenue × 0.30)
- Ranks sellers by revenue

##### Load and Join Data

In [0]:
# Load tables
df_fact = spark.table("olist_ecommerce_project.gold.fact_orders")
df_order_items = spark.table("olist_ecommerce_project.silver.slv_order_items")
df_sellers = spark.table("olist_ecommerce_project.gold.dim_sellers")

# Join: fact → order_items (to get seller_id) → sellers
df_seller_data = (
    df_fact
    .join(df_order_items, on="order_id", how="inner")
    .join(df_sellers, on="seller_id", how="left")
)

print("Joined data rows:", df_seller_data.count())

#### Aggregate by Seller

In [0]:
# Step 1: Group by seller and calculate metrics
df_seller_agg = (
    df_seller_data
    .groupBy("seller_id", "seller_state", "seller_city")
    .agg(
        count(col("order_id")).alias("total_orders"),
        countDistinct("customer_id").alias("total_customers"),
        spark_sum("total_order_value").alias("gross_revenue"),
        spark_round(avg("total_order_value"), 2).alias("avg_order_value"),
        count(when(col("is_late") == False, 1)).alias("on_time_orders"),
        count(when(col("is_late").isNotNull(), 1)).alias("delivered_orders"),
        spark_round(avg("review_score"), 2).alias("avg_review_score"),
        count(when(col("review_score").isNotNull(), 1)).alias("total_reviews")
    )
)

print("Aggregated sellers:", df_seller_agg.count())
df_seller_agg.show(3, truncate=False)

#### Calculate On-Time Rate

In [0]:

# Step 2: Calculate on-time delivery percentage safely
df_seller_agg = (
    df_seller_agg
    .withColumn(
        "on_time_delivery_rate_pct",
        spark_round(
            try_divide(
                col("on_time_orders") * 100,
                col("delivered_orders")
            ),
            2
        )
    )
)

# Show results — sellers with zero delivered orders will have NULL
df_seller_agg.select(
    "seller_id",
    "total_orders",
    "delivered_orders",
    "on_time_orders",
    "on_time_delivery_rate_pct"
).show(5, truncate=False)

# Check how many sellers have no delivered orders
print("Sellers with NULL on_time rate (no deliveries):")
df_seller_agg.filter(col("on_time_delivery_rate_pct").isNull()).count()

####  Rank Sellers by Revenue

In [0]:
# Step 3: Rank sellers globally by revenue (highest first)
window_revenue = Window.orderBy(col("gross_revenue").desc())

df_seller_agg = (
    df_seller_agg
    .withColumn(
        "revenue_rank",
        rank().over(window_revenue)
    )
)

df_seller_agg.select(
    "seller_id",
    "gross_revenue",
    "revenue_rank"
).orderBy("revenue_rank").show(10, truncate=False)

#### Normalize Scores for Health Score

In [0]:
# Step 4: Normalize each metric to 0-100 scale
# on_time_rate: already 0-100
# rating: scale 1-5 to 0-100
# revenue: use percentile rank (0-100)

df_seller_scores = (
    df_seller_agg
    .withColumn(
        "on_time_norm_score",
        col("on_time_delivery_rate_pct")  # Already 0-100
    )
    .withColumn(
        "rating_norm_score",
        (col("avg_review_score") / 5.0) * 100  # Scale 1-5 to 0-100
    )
)

# For revenue normalization, use row_number to create percentile
window_all = Window.orderBy(col("gross_revenue"))
df_seller_scores = (
    df_seller_scores
    .withColumn(
        "revenue_percentile",
        (row_number().over(window_all) / count("*").over(Window.partitionBy())) * 100
    )
    .withColumnRenamed("revenue_percentile", "revenue_norm_score")
)

df_seller_scores.select(
    "seller_id",
    "on_time_norm_score",
    "rating_norm_score",
    "revenue_norm_score"
).show(5, truncate=False)

#### Calculate Composite Health Score

In [0]:
# Step 5: Calculate seller_health_score
# Weighted composite: on_time 40% + rating 30% + revenue 30%

df_seller_health = (
    df_seller_scores
    .withColumn(
        "seller_health_score",
        spark_round(
            (col("on_time_norm_score") * 0.40) +
            (col("rating_norm_score") * 0.30) +
            (col("revenue_norm_score") * 0.30),
            2
        )
    )
)

print("Health scores calculated")
df_seller_health.select(
    "seller_id",
    "gross_revenue",
    "on_time_norm_score",
    "rating_norm_score",
    "revenue_norm_score",
    "seller_health_score"
).orderBy("seller_health_score", ascending=False).show(10, truncate=False)

#### Final Select and Write to Gold

In [0]:
# Step 6: Select final columns for the mart
df_mart_seller = (
    df_seller_health.select(
        "seller_id",
        "seller_state",
        "seller_city",
        "total_orders",
        "total_customers",
        "gross_revenue",
        "avg_order_value",
        "on_time_delivery_rate_pct",
        "avg_review_score",
        "total_reviews",
        "revenue_rank",
        "seller_health_score"
    )
    .orderBy("revenue_rank")
)

print("mart_seller_scorecard rows:", df_mart_seller.count())
df_mart_seller.show(5, truncate=False)

# Write to Gold
(
    df_mart_seller.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.gold.mart_seller_scorecard")
)

print("mart_seller_scorecard written successfully")